<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module4_Labs/Lab13.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 13 — H₂ Potential-Energy Curve and Bond-Length Optimization with VQE
**Quantum Optimization and Simulation — VQE Laboratory Series**

Lab 5 found the ground-state energy at one fixed H–H distance. This lab repeats that calculation at several distances to build a potential-energy curve and estimate the equilibrium bond length.

For each bond length \(R\), VQE first optimizes the circuit parameter \(\theta\). We then compare the best energies across \(R\).

**Suggested use:** guided lab. The geometry scan can take longer than the earlier labs.

> The required part uses an ideal Aer simulation. A fake-device noise model and real IBM hardware are included near the end as extensions.


## Learning objectives
1. Explain why changing molecular geometry changes the Hamiltonian.
2. Generate the H₂ Hamiltonian at several bond lengths.
3. Distinguish the inner VQE optimization over \(\theta\) from the outer search over \(R\).
4. Construct an H₂ potential-energy curve.
5. Estimate the equilibrium bond length.
6. Count the Estimator energy evaluations requested.
7. Compare ideal and fake-device-noise results at one selected geometry.
8. Optionally evaluate one optimized geometry on real IBM Quantum hardware.


In [ ]:
%pip -q install pylatexenc matplotlib
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" \
    "qiskit-algorithms~=0.4" "qiskit-nature~=0.8" "pyscf~=2.8"

%pip -q install "qiskit-ibm-runtime~=0.47"

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit.synthesis import LieTrotter
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.operators import FermionicOp

np.set_printoptions(precision=6, suppress=True)

SEED = 123
HARTREE_TO_EV = 27.211386245988


## Part A — Rebuild the Lab 5 four-qubit ansatz

We retain the same spin-orbital order used in Lab 5:

\[
q_0=\sigma_g\uparrow,\quad q_1=\sigma_u^*\uparrow,\quad
q_2=\sigma_g\downarrow,\quad q_3=\sigma_u^*\downarrow.
\]

The Hartree–Fock occupation is

\[
|1010\rangle_{q_0q_1q_2q_3},
\]

and the paired double excitation connects it to

\[
|0101\rangle_{q_0q_1q_2q_3}.
\]

Qiskit displays statevector bitstrings in the reverse order
\(q_3q_2q_1q_0\). Therefore:

| Physical order \(q_0q_1q_2q_3\) | Qiskit display \(q_3q_2q_1q_0\) |
|---|---|
| HF \(1010\) | `0101` |
| Double \(0101\) | `1010` |


In [ ]:
def hartree_fock_circuit():
    qc = QuantumCircuit(4)
    qc.x(0)
    qc.x(2)
    return qc


double_excitation = FermionicOp(
    {
        # bonding alpha/beta -> antibonding alpha/beta
        "+_1 +_3 -_2 -_0": 1.0,

        # de-excitation
        "+_0 +_2 -_3 -_1": -1.0,
    },
    num_spin_orbitals=4,
)

mapper = JordanWignerMapper()
jw_antihermitian = mapper.map(double_excitation).simplify()
double_generator = (1j * jw_antihermitian).simplify()

theta = Parameter("theta")

ansatz = hartree_fock_circuit()
ansatz.append(
    PauliEvolutionGate(
        double_generator,
        time=theta,
        synthesis=LieTrotter(reps=1),
    ),
    range(4),
)

display(ansatz.draw("mpl", fold=120))
print("Variational parameters:", ansatz.parameters)


## Part B — Generate \(H(R)\) for any bond length

Changing \(R\) changes the molecular orbitals, electronic integrals, nuclear
repulsion, and therefore the coefficients of the qubit Hamiltonian.

For every geometry, PySCF and Qiskit Nature construct

\[
H_{\mathrm{qubit}}(R)=\sum_j c_j(R)P_j.
\]

The nuclear-repulsion energy is added separately:

\[
E_{\mathrm{total}}(R)=E_{\mathrm{electronic}}(R)+E_{\mathrm{nuc}}(R).
\]


In [ ]:
def run_driver(R):
    """Run PySCF for H2 with bond length R in Angstrom."""
    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {R}",
        basis="sto3g",
        charge=0,
        spin=0,
    )
    return driver.run()


def hamiltonian_at(R):
    """Return the four-qubit electronic Hamiltonian and nuclear repulsion."""
    problem = run_driver(R)
    fermionic_hamiltonian = problem.hamiltonian.second_q_op()
    qubit_hamiltonian = mapper.map(fermionic_hamiltonian).simplify()

    return qubit_hamiltonian, float(problem.nuclear_repulsion_energy)


In [ ]:
# Check several representative geometries before running the full scan.
for R in [0.50, 0.74, 1.00, 1.50, 2.50]:
    H_R, E_nuc_R = hamiltonian_at(R)
    print(
        f"R = {R:4.2f} A | Pauli terms = {len(H_R):2d} "
        f"| nuclear repulsion = {E_nuc_R:.6f} Ha"
    )


### YOUR TURN 1

Why must the Hamiltonian be rebuilt whenever \(R\) changes, rather than using
one fixed Hamiltonian for the entire potential-energy curve?

<details>
<summary><b>Suggested answer</b></summary>

Changing the nuclear positions changes the electron–nuclear attraction,
nuclear–nuclear repulsion, molecular orbitals, and one- and two-electron
integrals. Consequently, the coefficients of the electronic Hamiltonian depend
on \(R\).

</details>


## Part C — Define the electronic-energy evaluator

`EstimatorV2` evaluates

\[
\langle\psi(\theta)|H(R)|\psi(\theta)\rangle.
\]

The molecular total energy is obtained by adding the classical
nuclear-repulsion energy.


In [ ]:
backend = AerSimulator(method="statevector")
estimator = EstimatorV2(
    options={
        "backend_options": {
            "method": "statevector",
            "seed_simulator": SEED,
        }
    }
)

ESTIMATOR_EXECUTIONS = 0

def reset_execution_counter():
    global ESTIMATOR_EXECUTIONS
    ESTIMATOR_EXECUTIONS = 0

def run_estimator(qc, observable):
    """Evaluate an observable with Aer EstimatorV2."""
    global ESTIMATOR_EXECUTIONS

    tqc = transpile(
        qc,
        backend=backend,
        optimization_level=1,
        seed_transpiler=SEED,
    )

    mapped_observable = observable.apply_layout(tqc.layout)

    result = estimator.run([(tqc, mapped_observable)]).result()
    ESTIMATOR_EXECUTIONS += 1

    return float(np.real(result[0].data.evs))

def energy(theta_value, H_R, E_nuc_R):
    """Total H2 energy at fixed R and fixed variational parameter theta."""
    bound = ansatz.assign_parameters({theta: float(theta_value)})
    electronic_energy = run_estimator(bound, H_R)
    return electronic_energy + E_nuc_R


## Part D — Inner loop: run VQE at one geometry

At a fixed \(R\), COBYLA varies only the circuit parameter \(\theta\):

\[
\theta^*(R)=\arg\min_\theta E(\theta;R).
\]

The geometry does not change during this inner optimization.


In [ ]:
def run_vqe(H_R, E_nuc_R, theta0=0.0, maxiter=60):
    """Optimize the one-parameter ansatz at one fixed geometry."""
    history = []

    def objective(theta_array):
        theta_value = float(np.atleast_1d(theta_array)[0])
        total = energy(theta_value, H_R, E_nuc_R)
        history.append((theta_value, total))
        return total

    result = minimize(
        objective,
        x0=np.array([theta0], dtype=float),
        method="COBYLA",
        options={
            "maxiter": maxiter,
            "rhobeg": 0.2,
            "tol": 1e-8,
        },
    )

    return float(result.x[0]), float(result.fun), result, history


In [ ]:
reset_execution_counter()

R_test = 0.74
H_test, E_nuc_test = hamiltonian_at(R_test)
theta_test, E_test, result_test, history_test = run_vqe(
    H_test,
    E_nuc_test,
    theta0=0.0,
)

E_hf_test = energy(0.0, H_test, E_nuc_test)
E_exact_test = np.linalg.eigvalsh(H_test.to_matrix())[0] + E_nuc_test

print(f"R = {R_test:.2f} A")
print(f"HF energy:                {E_hf_test:.8f} Ha")
print(f"VQE energy:               {E_test:.8f} Ha")
print(f"Exact diagonalization:    {E_exact_test:.8f} Ha")
print(f"Optimal theta:            {theta_test:.8f}")


print("Estimator energy evaluations for this VQE:", ESTIMATOR_EXECUTIONS)

## Part E — Outer loop: construct the potential-energy curve

We now repeat the inner VQE calculation over a range of bond lengths.

A **warm start** is used: the optimized \(\theta\) at one geometry becomes the
initial value for the next nearby geometry. Neighboring geometries normally
have similar ground states, so this can reduce optimizer work.


In [ ]:
def probability_in_physical_order(state, physical_bits):
    """
    Return the probability of a state labeled in q0,q1,q2,q3 order.

    Qiskit probabilities_dict() labels states in q3,q2,q1,q0 order.
    """
    qiskit_label = physical_bits[::-1]
    return float(state.probabilities_dict().get(qiskit_label, 0.0))


reset_execution_counter()
t0 = time.time()

Rs = np.round(np.arange(0.35, 2.51, 0.05), 2)

E_curve = []
E_hf_curve = []
E_exact_curve = []
double_weight_curve = []
theta_curve = []
function_evaluations = []

theta_guess = 0.0

for R in Rs:
    H_R, E_nuc_R = hamiltonian_at(R)

    theta_opt, E_vqe, result_R, history_R = run_vqe(
        H_R,
        E_nuc_R,
        theta0=theta_guess,
    )

    # Warm-start the next nearby geometry.
    theta_guess = theta_opt

    E_hf = energy(0.0, H_R, E_nuc_R)
    E_exact = np.linalg.eigvalsh(H_R.to_matrix())[0] + E_nuc_R

    optimized_circuit = ansatz.assign_parameters({theta: theta_opt})
    optimized_state = Statevector.from_instruction(optimized_circuit)

    # Doubly excited state: |0101> in physical q0,q1,q2,q3 order.
    w_double = probability_in_physical_order(
        optimized_state,
        physical_bits="0101",
    )

    E_curve.append(E_vqe)
    E_hf_curve.append(E_hf)
    E_exact_curve.append(E_exact)
    double_weight_curve.append(w_double)
    theta_curve.append(theta_opt)
    function_evaluations.append(len(history_R))

E_curve = np.asarray(E_curve)
E_hf_curve = np.asarray(E_hf_curve)
E_exact_curve = np.asarray(E_exact_curve)
double_weight_curve = np.asarray(double_weight_curve)
theta_curve = np.asarray(theta_curve)
function_evaluations = np.asarray(function_evaluations)

elapsed = time.time() - t0
max_error_mHa = 1000 * np.max(np.abs(E_curve - E_exact_curve))

print(f"Completed {len(Rs)} geometries in {elapsed:.1f} s")
print(f"Maximum |VQE - exact diagonalization| = {max_error_mHa:.5f} mHa")
print(f"Average VQE energy evaluations per geometry = {function_evaluations.mean():.1f}")


print("Estimator energy evaluations in coarse scan:", ESTIMATOR_EXECUTIONS)

## Part F — Refine the equilibrium bond length

The \(0.05\) Å coarse grid locates the minimum only approximately. We therefore
scan a smaller interval around the best coarse point using a \(0.01\) Å grid.

For H₂, this simple one-dimensional scan is transparent and sufficient for an
instructional laboratory. A general molecule would require a multidimensional
geometry optimizer.


In [ ]:
reset_execution_counter()

j_coarse = int(np.argmin(E_curve))
R_coarse = float(Rs[j_coarse])

R_fine = np.round(
    np.arange(R_coarse - 0.04, R_coarse + 0.041, 0.01),
    2,
)

E_fine = []
theta_fine = []

# Start the fine scan from the best parameter found on the coarse grid.
theta_guess = float(theta_curve[j_coarse])

for R in R_fine:
    H_R, E_nuc_R = hamiltonian_at(R)
    theta_opt, E_vqe, _, _ = run_vqe(
        H_R,
        E_nuc_R,
        theta0=theta_guess,
    )
    theta_guess = theta_opt

    theta_fine.append(theta_opt)
    E_fine.append(E_vqe)

E_fine = np.asarray(E_fine)
theta_fine = np.asarray(theta_fine)

j_fine = int(np.argmin(E_fine))
R_eq = float(R_fine[j_fine])
E_min = float(E_fine[j_fine])

print("Fine scan:")
for R, E_R in zip(R_fine, E_fine):
    print(f"R = {R:.2f} A   E = {E_R:.8f} Ha")

print(f"\nEstimated equilibrium bond length R_eq = {R_eq:.2f} A")
print(f"Experimental H2 bond length is approximately 0.741 A")
print(f"Minimum VQE total energy E_0 = {E_min:.8f} Ha")
print(f"Minimum VQE total energy E_0 = {E_min * HARTREE_TO_EV:.4f} eV")


print("Estimator energy evaluations in fine scan:", ESTIMATOR_EXECUTIONS)

## Part G — Plot and interpret the results

The left graph compares three methods:

- **Hartree–Fock:** the uncorrelated reference state, \(\theta=0\);
- **VQE:** the optimized one-parameter paired-excitation ansatz;
- **Exact diagonalization in STO-3G:** the lowest eigenvalue of the same
  finite-basis qubit Hamiltonian.

“Exact” here means exact for the selected STO-3G model—not the exact physical
energy of real H₂.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

ax[0].plot(
    Rs,
    E_hf_curve,
    "s--",
    ms=3,
    label=r"Hartree-Fock ($\theta=0$)",
)
ax[0].plot(
    Rs,
    E_curve,
    "o-",
    ms=3,
    label="VQE (4-qubit paired double)",
)
ax[0].plot(
    Rs,
    E_exact_curve,
    "k:",
    lw=2,
    label="Exact diagonalization in STO-3G",
)
ax[0].plot(R_eq, E_min, "r*", ms=14, label="Estimated minimum")
ax[0].set_xlabel("H-H distance R (Angstrom)")
ax[0].set_ylabel("Total energy (Hartree)")
ax[0].set_title("H2 potential-energy curve")
ax[0].grid(True)
ax[0].legend(fontsize=8)

ax[1].plot(
    Rs,
    100 * double_weight_curve,
    "o-",
    ms=3,
)
ax[1].axvline(R_eq, color="k", ls=":", lw=1)
ax[1].set_xlabel("H-H distance R (Angstrom)")
ax[1].set_ylabel(r"Weight of double configuration $|0101\rangle$ (%)")
ax[1].set_title("Growth of static correlation")
ax[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Approximate dissociation energy using the final sampled point, R = 2.50 A.
# This is not the true R -> infinity limit.
D_sampled_eV = (E_curve[-1] - E_min) * HARTREE_TO_EV

print(f"Estimated R_eq:                 {R_eq:.2f} A")
print(f"Minimum energy:                {E_min:.8f} Ha")
print(f"Sampled dissociation estimate: {D_sampled_eV:.3f} eV")
print("Experimental dissociation energy is approximately 4.75 eV.")
print("The STO-3G basis and restricted ansatz are instructional, not quantitative.")

j_near = int(np.argmin(np.abs(Rs - R_eq)))

print(
    f"\nDouble-configuration weight near equilibrium "
    f"(R={Rs[j_near]:.2f} A): "
    f"{100 * double_weight_curve[j_near]:.2f}%"
)
print(
    f"Double-configuration weight at R={Rs[-1]:.2f} A: "
    f"{100 * double_weight_curve[-1]:.2f}%"
)


### YOUR TURN 2

Why does the doubly excited configuration become more important as the H–H
bond is stretched?

<details>
<summary><b>Suggested answer</b></summary>

Near equilibrium, one Hartree–Fock determinant provides a good leading
description, and the doubly excited configuration is mainly a small correlation
correction. At long bond length, the bonding and antibonding configurations
approach similar importance. A single determinant can no longer represent the
ground state well, so a superposition of configurations is needed. This is
called static or strong correlation.

</details>


## Part H — Visualize the optimized parameter

The variational parameter is optimized independently for every geometry.
Its change with \(R\) reflects the changing mixture of the Hartree–Fock and
doubly excited configurations.


In [ ]:
plt.figure(figsize=(6.5, 3.8))
plt.plot(Rs, theta_curve, "o-", ms=3)
plt.axvline(R_eq, color="k", ls=":", lw=1)
plt.xlabel("H-H distance R (Angstrom)")
plt.ylabel(r"Optimal double-excitation parameter $\theta^*(R)$")
plt.title("Geometry dependence of the optimized VQE parameter")
plt.grid(True)
plt.show()


## Optional extension — Compare cold starts and warm starts

Modify Part E so that every geometry starts from `theta0=0.0`. Compare the
number of energy evaluations with the warm-start calculation.

Questions:

1. Does warm starting reduce the average number of optimizer evaluations?
2. Does it change the final energy curve?
3. Why should nearby geometries usually have nearby optimal parameters?


## Part I — Fake-device noise at the estimated equilibrium geometry

Running the entire bond-length curve with a device noise model would be slow and would hide the main geometry lesson. Instead, we take the optimized geometry found above and ask a focused question:

> How much does a realistic fake-device noise model change the energy at \(R_{\rm eq}\)?

A noisy value does not have to equal the ideal VQE value. More shots can reduce statistical variation, but systematic gate and readout errors can remain.


In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit_aer.noise import NoiseModel

fake_backend = FakeVigoV2()
fake_noise = NoiseModel.from_backend(fake_backend)

noisy_backend = AerSimulator(noise_model=fake_noise)
noisy_estimator = EstimatorV2(
    options={
        "backend_options": {
            "noise_model": fake_noise,
            "seed_simulator": SEED,
        }
    }
)

def run_noisy_estimator(qc, observable):
    tqc = transpile(
        qc,
        backend=noisy_backend,
        basis_gates=fake_noise.basis_gates,
        optimization_level=3,
        seed_transpiler=SEED,
    )
    mapped_observable = observable.apply_layout(tqc.layout)
    result = noisy_estimator.run([(tqc, mapped_observable)]).result()
    return float(np.real(result[0].data.evs))

H_eq, E_nuc_eq = hamiltonian_at(R_eq)
theta_eq = float(theta_fine[j_fine])
bound_eq = ansatz.assign_parameters({theta: theta_eq})

ideal_eq = run_estimator(bound_eq, H_eq) + E_nuc_eq
noisy_eq = run_noisy_estimator(bound_eq, H_eq) + E_nuc_eq

print("R_eq:", R_eq, "Angstrom")
print("Ideal VQE energy:", ideal_eq, "Ha")
print("Fake-device noisy energy:", noisy_eq, "Ha")
print("Difference:", noisy_eq - ideal_eq, "Ha")


### Interpreting the difference

A fake-device simulation adds gate and readout errors. The energy can therefore move away from the ideal minimum.

Increasing shots is useful when the result jumps around from run to run, because statistical uncertainty scales roughly as \(1/\sqrt{N}\). But more shots do **not** erase systematic device noise.

Practical ways to improve a hardware experiment include:

- more shots for better precision;
- better transpilation to reduce unnecessary gates;
- measurement-error mitigation;
- more advanced error-mitigation methods such as zero-noise extrapolation.

For this introductory lab, we stop at the comparison rather than implementing a full mitigation stack.


## Optional — Evaluate one geometry on real IBM Quantum hardware

This is intentionally a **single-point** hardware experiment, not a full geometry optimization on a QPU. The simulator first finds \(R_{eq}\) and \(	heta_{eq}\); hardware is then used only to estimate the energy of that optimized circuit.

Real-device results will normally differ from the ideal simulation because of finite sampling and hardware noise.


In [ ]:
# OPTIONAL REAL-HARDWARE TEMPLATE — requires IBM Quantum access.

# from qiskit_ibm_runtime import QiskitRuntimeService
# from qiskit_ibm_runtime import EstimatorV2 as RuntimeEstimatorV2
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#
# service = QiskitRuntimeService()
# real_backend = service.least_busy(
#     operational=True,
#     simulator=False,
#     min_num_qubits=4,
# )
#
# pm = generate_preset_pass_manager(
#     backend=real_backend,
#     optimization_level=3,
# )
# isa_circuit = pm.run(bound_eq)
# isa_observable = H_eq.apply_layout(isa_circuit.layout)
#
# runtime_estimator = RuntimeEstimatorV2(mode=real_backend)
#
# # A finite precision requests a finite amount of sampling.
# job = runtime_estimator.run(
#     [(isa_circuit, isa_observable)],
#     precision=0.02,
# )
# hardware_electronic = float(np.real(job.result()[0].data.evs))
# hardware_total = hardware_electronic + E_nuc_eq
#
# print("Backend:", real_backend.name)
# print("Real-hardware total energy:", hardware_total, "Ha")
# print("Estimator requests submitted: 1")


## Final reflection

1. What is optimized in the inner loop, and what is varied in the outer loop?
2. Why does the Hamiltonian have to be rebuilt when the H–H distance changes?
3. Why is exact diagonalization practical here but difficult for large active spaces?
4. Why can a fake-device or real-hardware energy differ from the ideal VQE energy?
5. What does increasing the number of shots improve, and what does it not fix?
6. Why is a single-point hardware check more practical than running the full geometry scan on a QPU?
